# SHIELD PHARMA-HYBRID v21.0 - FINAL PRODUCTION LAUNCHER
## AntiGravity 통합 실행 및 시스템 무결성 검증 스크립트

**포함 모듈:**
- PHARMA-HYBRID 메인 대시보드 (Streamlit)
- AI 데이터셋 (학습/검증/테스트) : `pharma_ai_antigravity.py`
- 환자 처방전 데이터셋 (31건) : `prescription_index.json`

In [ ]:
import os
import sys
import time
import subprocess
from datetime import datetime
from pathlib import Path

# 윈도우 터미널 인코딩 대응 (UTF-8 강제)
if sys.platform == "win32":
    import io
    # Note: Jupyter 환경에서는 sys.stdout 재정의가 불필요할 수 있습니다.
    # sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding='utf-8')
    # sys.stderr = io.TextIOWrapper(sys.stderr.buffer, encoding='utf-8')

In [ ]:
# ANSI 색상 코드
class Color:
    CYAN = '\033[96m'
    GREEN = '\033[92m'
    YELLOW = '\033[93m'
    RED = '\033[91m'
    BOLD = '\033[1m'
    END = '\033[0m'

In [ ]:
def print_banner():
    banner = f"""
{Color.CYAN}{Color.BOLD}
    ======================================================================
    SHIELD PHARMA-HYBRID v21.0 : PRODUCTION STANDALONE
    ======================================================================
    [SYSTEM] : Strategic Clinical Decision Support System
    [DATASET]: AntiGravity AI Dataset v1.0 (Train 21 / Val 6 / Test 4)
    [STATUS] : 3-Column Modernized Layout Active
    [TIME]   : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
    ======================================================================
{Color.END}"""
    print(banner)

In [ ]:
def check_dataset():
    """AI 학습/검증/테스트 데이터셋 로드 및 요약 출력."""
    print(f"{Color.YELLOW}[DATASET] AntiGravity 데이터셋 로딩 중...{Color.END}")
    try:
        from pharma_ai_antigravity import (
            TRAINING_SET, VALIDATION_SET, TEST_SET,
            DRUG_KB, verify_integrity, run_dataset_stats, export_to_json
        )
        total = len(TRAINING_SET) + len(VALIDATION_SET) + len(TEST_SET)
        print(f"{Color.GREEN}  학습(Train) : {len(TRAINING_SET):>2}건{Color.END}")
        print(f"{Color.GREEN}  검증(Val)   : {len(VALIDATION_SET):>2}건{Color.END}")
        print(f"{Color.GREEN}  테스트(Test): {len(TEST_SET):>2}건{Color.END}")
        print(f"{Color.CYAN}  합계        : {total:>2}건 | 약물 DB: {len(DRUG_KB)}종{Color.END}")

        # 테스트셋 환자 목록 표시
        print(f"\n{Color.YELLOW}  [TEST] 신규·복합 케이스:{Color.END}")
        for rx in TEST_SET:
            disease_short = rx.diseases[0][:20] if rx.diseases else '-'
            meds_count = len(rx.medications)
            severity = '⚠️ HIGH' if rx.interaction_flags and any('⚠️' in f for f in rx.interaction_flags) else 'LOW'
            print(f"    {rx.sample_id} {rx.name}({rx.age}세) | {disease_short} | 약 {meds_count}종 | {severity}")

        # JSON 내보내기 (백엔드 동기화)
        export_to_json('pharma_antigravity_dataset.json')
        verify_integrity()
        return True
    except ImportError:
        print(f"{Color.RED}  ❌ pharma_ai_antigravity.py 없음 — 데이터셋 건너뜜{Color.END}")
        return True
    except Exception as e:
        print(f"{Color.RED}  ❌ 데이터셋 오류: {e}{Color.END}")
        return True

In [ ]:
def check_system():
    print(f"{Color.YELLOW}[1/4] 시스템 무결성 검사 중...{Color.END}")
    essential_files = [
        "전부_코드화_데이터통합시스템.py",
        "gemini_ai_engine.py",
        "drug_info_complete_db.py",
        "disease_knowledge_db.py",
        "real_patient_data.json",
        "prescription_index.json",
        "pharma_ai_antigravity.py",
    ]

    missing = []
    for f in essential_files:
        if not Path(f).exists():
            missing.append(f)

    optional = ["pharma_antigravity_dataset.json"]
    for f in optional:
        status = "✅" if Path(f).exists() else "⚠️ (미생성)"
        print(f"  {status} {f}")

    if missing:
        critical = [f for f in missing if 'antigravity' not in f and 'prescription_index' not in f]
        if critical:
            print(f"{Color.RED}❌ 필수 파일 누락: {', '.join(critical)}{Color.END}")
            return False
        else:
            print(f"{Color.YELLOW}⚠️  보조 파일 누락 (계속 진행): {', '.join(missing)}{Color.END}")

    print(f"{Color.GREEN}✅ 파일 무결성 확인 완료{Color.END}")
    return True

In [ ]:
def check_environment():
    print(f"{Color.YELLOW}[2/4] 실행 환경 구성 확인 중...{Color.END}")
    try:
        import streamlit
        import google.generativeai
        print(f"{Color.GREEN}✅ 핵심 라이브러리 로드 성공 (Streamlit, Gemini SDK){Color.END}")
    except ImportError as e:
        print(f"{Color.RED}❌ 라이브러리 누락: {e}{Color.END}")
        return False
    return True

In [ ]:
def launch_dashboard():
    print(f"{Color.YELLOW}[4/4] PHARMA-HYBRID 대시보드 기동 중...{Color.END}")
    print(f"{Color.CYAN}----------------------------------------------------------------------")
    print(f"  브라우저에서 대시보드가 열립니다. (포트: 8503)")
    print(f"----------------------------------------------------------------------{Color.END}")

    # Jupyter에서 Streamlit을 직접 실행하는 것은 주의가 필요합니다.
    # 별도의 터미널에서 실행하거나, subprocess를 비동기로 실행해야 할 수 있습니다.
    cmd = ["streamlit", "run", "전부_코드화_데이터통합시스템.py", "--server.port", "8503"]
    
    print(f"[COMMAND] {' '.join(cmd)}")
    # 주석 해제 시 실행:
    # subprocess.run(cmd, check=True)

In [ ]:
def main():
    print_banner()
    if not check_system():
        print(f"{Color.RED}시스템 가동 실패. 로그를 확인하세요.{Color.END}")
        return

    print(f"\n{Color.YELLOW}[3/4] AI 데이터셋 검증 중...{Color.END}")
    check_dataset()

    if not check_environment():
        print(f"{Color.RED}환경 설정 실패.{Color.END}")
        return

    time.sleep(1)
    launch_dashboard()

if __name__ == "__main__":
    main()